# Model Training — Grid Search MLP · RNN · LSTM

## Deep Learning Project | Bitcoin Volatility Forecasting

**Estrategia:** Grid search sobre fold 0, lag=14d para seleccionar arquitectura por tipo.  
La ganadora se evalúa en los **5 folds completos** con las métricas del enunciado.

| Componente | Detalle |
|---|---|
| Tipos | MLP · RNN · LSTM |
| Neuronas/capa | 32 · 64 · 128 · 256 |
| Máx. capas | 3 |
| Dropout | on/off por capa — rate=0.3 |
| Selección | `gap_ratio < 0.15`, ordenado por `best_val_rmse` |
| Evaluación final | 5 folds × 4 lags × métricas + BDS test |


## Configuración del entorno TensorFlow y verificación de hardware

Se configura el entorno de TensorFlow para el entrenamiento del modelo, estableciendo el nivel de verbosidad mínimo para suprimir mensajes informativos y de advertencia no críticos. Se imprime la versión de TensorFlow instalada y se listan todos los dispositivos físicos detectados (CPU, GPU, TPU disponibles en el sistema). Se verifica específicamente la presencia de GPUs: si se detecta al menos una, se reporta su nombre y cantidad; en caso contrario, se advierte al usuario que el entrenamiento se ejecutará en CPU, lo que implicará tiempos de procesamiento significativamente mayores (especialmente durante la búsqueda de hiperparámetros o grid search). Se solicita confirmación al usuario para continuar sin GPU, evitando ejecuciones accidentales que podrían extenderse por varias horas. Si el usuario no confirma, se cancela la ejecución del programa.

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf

print(f"TensorFlow version: {tf.__version__}")
print()

devices = tf.config.list_physical_devices()
print("Dispositivos detectados:")
for d in devices:
    print(f" {d}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\nGPU ACTIVA — {len(gpus)} dispositivo(s):")
    for g in gpus:
        print(f" {g.name}")
else:
    print("\nSin GPU detectada — corriendo en CPU")
    print(" Si tienes DirectML instalado, verifica que el kernel sea 'Deep Learning (GPU)'")
    print(" El grid search tardará varias horas en CPU.")
    respuesta = input("\n¿Deseas continuar de todas formas? (s/n): ")
    if respuesta.lower() != 's':
        raise SystemExit("Ejecución cancelada. Configura la GPU antes de continuar.")


TensorFlow version: 2.10.0

Dispositivos detectados:
  PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')
  PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
  PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')

✅ GPU ACTIVA — 2 dispositivo(s):
   /physical_device:GPU:0
   /physical_device:GPU:1


## Importación de librerías y configuración de reproducibilidad

Se importan las librerías necesarias para el entrenamiento, evaluación y persistencia del modelo MLP, incluyendo `pickle` para carga de artefactos, `joblib` para serialización eficiente de modelos entrenados, y `scipy.stats` para pruebas estadísticas sobre residuos. Se fijan las semillas aleatorias tanto de TensorFlow como de NumPy con el valor 42, garantizando la reproducibilidad total de los experimentos (resultados idénticos en ejecuciones sucesivas). Se suprimen mensajes de advertencia no críticos para mantener la salida limpia durante el entrenamiento. Finalmente, se crean los directorios `results/` (para almacenar métricas, gráficos y logs de entrenamiento) y `app/` (para exportar modelos y artefactos destinados al despliegue en producción), verificando que existan sin errores si ya están creados.

In [ ]:
import pickle, warnings, itertools
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from scipy import stats

tf.random.set_seed(42)
np.random.seed(42)

os.makedirs('results', exist_ok=True)
os.makedirs('app', exist_ok=True)

print("Imports completos")

✅ Imports completos


## 1. Carga del artefacto features.pkl y selección de configuración para grid search

Se carga el artefacto `features.pkl` previamente generado y se extraen sus componentes principales: los splits de validación temporal para cada lag (`splits`), la lista de rezagos evaluados (`LAGS_LIST`), el horizonte de predicción (`N_STEPS_FORECAST`), el nombre de la columna objetivo (`TARGET_COL`), la serie temporal completa de volatilidad (`time_series`) y las fechas asociadas (`dates`). Se define una configuración específica para realizar la búsqueda de hiperparámetros (grid search): se selecciona el lag de 14 días como compromiso entre contexto temporal suficiente y tamaño muestral adecuado, y el fold 0 como partición representativa para el ajuste inicial. Esta selección permite explorar el espacio de hiperparámetros de manera eficiente sin ejecutar el grid search sobre todas las combinaciones de lag y fold, reduciendo drásticamente el tiempo de cómputo. Finalmente, se imprime un resumen de la configuración seleccionada.

In [ ]:
with open('features.pkl', 'rb') as f:
    data = pickle.load(f)

splits           = data['splits']
LAGS_LIST        = data['lags_list']
N_STEPS_FORECAST = data['n_steps_forecast']
TARGET_COL       = data['target_col']
time_series      = data['time_series']
dates            = data['dates']

GRID_LAG  = 14
GRID_FOLD = 0

print(f"Lags: {LAGS_LIST}")
print(f"Horizonte: {N_STEPS_FORECAST} dias")
print(f"Grid search: lag={GRID_LAG}d, fold={GRID_FOLD}")


Lags     : [7, 14, 21, 28]
Horizonte: 7 dias
Grid search: lag=14d, fold=0


## 2. Definición de funciones de evaluación y test de no linealidad de BDS

Se implementan las funciones necesarias para la evaluación del modelo y el análisis de residuos. Se definen cuatro métricas de error fundamentales: MAPE (Error Absoluto Porcentual Medio), MAE (Error Absoluto Medio), RMSE (Raíz del Error Cuadrático Medio) y MSE (Error Cuadrático Medio). Adicionalmente, se crea la función `metrics_per_horizon` que desagrega estas métricas para cada uno de los 7 días del horizonte de predicción (h1 a h7), calculando también promedios globales para resumir el desempeño general del modelo. Finalmente, se implementa el test de Brock-Dechert-Scheinkman (BDS), una prueba estadística no lineal que evalúa si los residuos del modelo son independientes e idénticamente distribuidos (i.i.d.) contra la alternativa de estructura no lineal remanente. El test se calcula para dimensiones de embebido de 2 a 4, utilizando una escala del umbral (epsilon) igual a 0.7 veces la desviación estándar de los residuos, y reporta tanto el estadístico como el p-valor para cada dimensión. Estas métricas y tests permitirá diagnosticar la calidad del ajuste y detectar si el MLP logra capturar adecuadamente la dependencia temporal presente en la volatilidad de Bitcoin.

In [ ]:
def mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask]-y_pred[mask])/y_true[mask]))*100

def mae(y_true, y_pred):  return np.mean(np.abs(y_true - y_pred))
def rmse(y_true, y_pred): return np.sqrt(np.mean((y_true-y_pred)**2))
def mse(y_true, y_pred):  return np.mean((y_true-y_pred)**2)

def metrics_per_horizon(y_true, y_pred):
    H = y_true.shape[1]
    result = {}
    for h in range(H):
        yt, yp = y_true[:,h], y_pred[:,h]
        result[f'h{h+1}'] = {
            'MAPE': round(mape(yt,yp),4), 'MAE':  round(mae(yt,yp),6),
            'RMSE': round(rmse(yt,yp),6), 'MSE':  round(mse(yt,yp),8),
        }
    result['mean'] = {
        k: round(np.mean([result[f'h{h+1}'][k] for h in range(H)]),6)
        for k in ['MAPE','MAE','RMSE','MSE']
    }
    return result

def bds_test(residuals, max_dim=2, eps_scale=0.7):
    x = np.asarray(residuals, dtype=float); x = x[~np.isnan(x)]
    n = len(x); eps = eps_scale * np.std(x)
    def c1_fn(arr):
        cnt = 0
        for i in range(len(arr)-1): cnt += np.sum(np.abs(arr[i+1:]-arr[i])<=eps)
        return 2*cnt/(n*(n-1))
    c1 = c1_fn(x); results = {}
    for m in range(2, max_dim+1):
        em = np.column_stack([x[i:n-m+i+1] for i in range(m)]); nr = len(em)
        cnt_m = 0
        for i in range(nr-1): cnt_m += np.sum(np.max(np.abs(em[i+1:]-em[i]),axis=1)<=eps)
        cm = 2*cnt_m/(nr*(nr-1))
        k_sum = sum(c1**(2*(m-j)) for j in range(1,m))
        var = (4/n)*(k_sum+(m-1)**2*c1**(2*m))
        stat = (cm-c1**m)/np.sqrt(max(var,1e-12))
        results[m] = {'statistic': round(float(stat),4),
                      'p_value': round(float(2*(1-stats.norm.cdf(abs(stat)))),4)}
    return results

print("Metricas y BDS listos")


✅ Metricas y BDS listos


## 3. Definición del espacio de búsqueda para grid search de arquitectura MLP

Se define el espacio de hiperparámetros a explorar en la búsqueda sistemática de la mejor arquitectura de red neuronal. Se especifican cuatro tamaños de capa oculta posibles (32, 64, 128 y 256 neuronas), con un máximo de 3 capas ocultas, generando todas las combinaciones posibles de arquitecturas (desde una sola capa de 32 neuronas hasta tres capas con 256 neuronas cada una). Se establece una tasa de dropout fija del 30% para regularización, aunque se exploran todas las combinaciones posibles de aplicación de dropout por capa (por ejemplo, aplicar dropout solo en la primera capa, solo en la segunda, en todas o en ninguna). Se fijan hiperparámetros de entrenamiento constantes para toda la búsqueda: 50 épocas como máximo, tamaño de lote de 32 muestras, y paciencia de 10 épocas para early stopping. Se calcula el tamaño total del espacio de búsqueda: el número de arquitecturas únicas multiplicado por las configuraciones de dropout por arquitectura, multiplicado por 3 tipos de optimizadores o funciones de pérdida a evaluar (a definir posteriormente), resultando en el número total de entrenamientos que se ejecutarán durante el grid search.

In [5]:
neuronas     = [32, 64, 128, 256]
max_capas    = 3
DROPOUT_RATE = 0.3
EPOCHS       = 50
BATCH_SIZE   = 32
PATIENCE     = 10

arquitecturas = []
for n_capas in range(1, max_capas+1):
    for comb in itertools.product(neuronas, repeat=n_capas):
        arquitecturas.append(comb)

def dropout_configs(n_capas):
    for mask in range(2**n_capas):
        yield tuple(bool((mask>>i)&1) for i in range(n_capas))

total_configs = sum(2**len(a) for a in arquitecturas)
print(f"Arquitecturas unicas : {len(arquitecturas)}")
print(f"Configs arch+dropout : {total_configs}")
print(f"Total grid (3 tipos) : {total_configs*3} entrenamientos")


Arquitecturas unicas : 84
Configs arch+dropout : 584
Total grid (3 tipos) : 1752 entrenamientos


**Desglose del espacio de búsqueda**

| Nivel | Cantidad | Explicación |
|-------|----------|-------------|
| **Arquitecturas únicas** | 84 | Combinaciones de capas ocultas con neuronas {32,64,128,256} y 1-3 capas |
| **Configuraciones con dropout** | 584 | 84 arquitecturas × (2^capas) combinaciones de dropout |
| **Total entrenamientos** | 1,752 | 584 × 3 (tipos adicionales a definir) |

---

**Composición de las 84 arquitecturas por número de capas**

| Capas | Combinaciones | Ejemplos |
|-------|---------------|----------|
| 1 capa | 4 | (32), (64), (128), (256) |
| 2 capas | 4×4 = 16 | (32,32), (32,64), ..., (256,256) |
| 3 capas | 4×4×4 = 64 | (32,32,32), (32,32,64), ..., (256,256,256) |
| **Total** | **84** | Verificado |

**Por tamaño de arquitectura:**
- **Pequeñas** (≤64 neuronas/capa): ~40 configuraciones
- **Medianas** (128 neuronas/capa): ~25 configuraciones
- **Grandes** (256 neuronas/capa): ~19 configuraciones

---

**Expansión por dropout (584 configuraciones)**

Para una arquitectura con `n` capas, existen `2^n` formas de aplicar dropout (cada capa puede tener dropout o no).

**Desglose por arquitectura:**

| Capas | Arquitecturas | Dropout por arq. | Total dropout configs |
|-------|---------------|------------------|----------------------|
| 1 capa | 4 | 2^1 = 2 | 4 × 2 = 8 |
| 2 capas | 16 | 2^2 = 4 | 16 × 4 = 64 |
| 3 capas | 64 | 2^3 = 8 | 64 × 8 = 512 |
| **Total** | 84 | — | **584** |

## 4. Definición de constructores de arquitecturas neuronales (MLP, RNN y LSTM)

Se implementan tres funciones constructoras que permiten instanciar dinámicamente diferentes tipos de arquitecturas de redes neuronales durante el grid search. La función `build_mlp` construye un perceptrón multicapa estándar con capas densas completamente conectadas, activación ReLU y dropout opcional por capa, finalizando con una capa de salida de dimensión igual al horizonte de predicción (7 días). La función `build_rnn` implementa una red recurrente simple (RNN) que procesa secuencias temporales respetando el orden de los rezagos, con capacidad para retener información de corto plazo y opción de dropout para regularización. La función `build_lstm` construye una red Long Short-Term Memory, una variante avanzada de RNN que incorpora compuertas de olvido y memoria celular, permitiendo capturar dependencias temporales de largo plazo sin sufrir el problema de desvanecimiento del gradiente. Las tres funciones comparten la misma interfaz: reciben la arquitectura (lista de neuronas por capa), una máscara de dropout (indicando qué capas aplican dropout), las dimensiones de entrada y salida (para MLP) o el número de timesteps (para RNN/LSTM), y compilan el modelo con optimizador Adam, función de pérdida MSE y la métrica RMSE para monitoreo durante el entrenamiento. Estos constructores serán invocados iterativamente durante el grid search para evaluar sistemáticamente las 1,752 configuraciones definidas previamente.

In [ ]:
def build_mlp(arch, dmask, input_dim, output_dim):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(input_dim,)))
    for idx, units in enumerate(arch):
        model.add(tf.keras.layers.Dense(units, activation='relu'))
        if dmask[idx]:
            model.add(tf.keras.layers.Dropout(DROPOUT_RATE))
    model.add(tf.keras.layers.Dense(output_dim))
    model.compile(optimizer='adam', loss='mse',
                  metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse')])
    return model

def build_rnn(arch, dmask, timesteps, output_dim):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(timesteps, 1)))
    for idx, units in enumerate(arch[:-1]):
        model.add(tf.keras.layers.SimpleRNN(units, return_sequences=True))
        if dmask[idx]:
            model.add(tf.keras.layers.Dropout(DROPOUT_RATE))
    model.add(tf.keras.layers.SimpleRNN(arch[-1], return_sequences=False))
    if dmask[-1]:
        model.add(tf.keras.layers.Dropout(DROPOUT_RATE))
    model.add(tf.keras.layers.Dense(output_dim))
    model.compile(optimizer='adam', loss='mse',
                  metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse')])
    return model

def build_lstm(arch, dmask, timesteps, output_dim):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(timesteps, 1)))
    for idx, units in enumerate(arch[:-1]):
        model.add(tf.keras.layers.LSTM(units, return_sequences=True))
        if dmask[idx]:
            model.add(tf.keras.layers.Dropout(DROPOUT_RATE))
    model.add(tf.keras.layers.LSTM(arch[-1], return_sequences=False))
    if dmask[-1]:
        model.add(tf.keras.layers.Dropout(DROPOUT_RATE))
    model.add(tf.keras.layers.Dense(output_dim))
    model.compile(optimizer='adam', loss='mse',
                  metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse')])
    return model

print("Constructores listos: MLP / RNN / LSTM")


✅ Constructores listos: MLP / RNN / LSTM


## 5. Implementación del grid search con reanudación y checkpoints
  
Se implementa la función principal `run_grid_search` que ejecuta la búsqueda sistemática de hiperparámetros sobre todas las configuraciones definidas (arquitectura × máscara de dropout × tipo de modelo). La función recibe como parámetros el tipo de modelo (MLP, RNN o LSTM), los conjuntos de entrenamiento y validación (features y target), la dimensión de entrada (lag) y salida (horizonte de 7 días), y parámetros de checkpointing. Internamente, genera una lista plana de todas las configuraciones a evaluar y soporta reanudación automática en caso de interrupción, utilizando un índice de inicio calculado a partir de resultados parciales previamente guardados. Para cada configuración, se limpia la sesión de TensorFlow (previniendo fugas de memoria), se fija la semilla aleatoria, se construye el modelo mediante los constructores definidos, y se entrena con early stopping (paciencia de 10 épocas). Se registran métricas clave: el mejor RMSE de validación, el RMSE final de entrenamiento, la brecha (gap) entre ambos, el número de épocas ejecutadas, y las curvas completas de entrenamiento y validación. Cada cierto número de configuraciones (por defecto cada 10), se guarda un checkpoint atómico: primero se escribe un archivo temporal, luego se reemplaza el archivo final mediante `os.replace`, garantizando que el checkpoint nunca quede corrupto incluso si el proceso se interrumpe durante la escritura. Se reporta el progreso en tiempo real con métricas de rendimiento (configuraciones por segundo). Finalmente, se retorna un DataFrame con todos los resultados para su análisis posterior. Esta implementación robusta permite ejecutar grid searches de gran escala (hasta 1,752 entrenamientos) de manera segura, reanudable y eficiente en memoria.

In [ ]:
import time, gc

def run_grid_search(model_type, X_tr, y_tr, X_val, y_val, input_lag, output_dim,
                    checkpoint_file=None, checkpoint_key=None,
                    partial_results=None, checkpoint_every=10):

    # Lista plana de configs: se indexa por posicion para reanudar facil.
    all_configs = [(arch, dmask)
                   for arch in arquitecturas
                   for dmask in dropout_configs(len(arch))]
    n_total = len(all_configs)

    resultados = list(partial_results) if partial_results else []
    start_idx  = len(resultados)

    if start_idx > 0:
        print(f"  [{model_type}] Reanudando desde config {start_idx}/{n_total} "
              f"({n_total - start_idx} restantes)")
    elif start_idx == n_total:
        print(f"  [{model_type}] Ya tenia {n_total}/{n_total} — nada que correr.")
        return pd.DataFrame(resultados)

    es = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=0)

    if model_type in ('RNN', 'LSTM'):
        X_tr_in  = X_tr.reshape(-1, input_lag, 1)
        X_val_in = X_val.reshape(-1, input_lag, 1)
    else:
        X_tr_in, X_val_in = X_tr, X_val

    block_start = time.time()

    for i in range(start_idx, n_total):
        arch, dmask = all_configs[i]

        tf.keras.backend.clear_session()
        tf.random.set_seed(42)

        if   model_type == 'MLP':  model = build_mlp(arch, dmask, input_lag, output_dim)
        elif model_type == 'RNN':  model = build_rnn(arch, dmask, input_lag, output_dim)
        else:                      model = build_lstm(arch, dmask, input_lag, output_dim)

        history = model.fit(
            X_tr_in, y_tr,
            validation_data=(X_val_in, y_val),
            epochs=EPOCHS, batch_size=BATCH_SIZE,
            callbacks=[es], verbose=0)

        train_rmse = history.history['rmse']
        val_rmse   = history.history['val_rmse']
        best_val   = float(np.min(val_rmse))
        last_train = float(train_rmse[-1])
        gap        = abs(last_train - best_val)
        gap_ratio  = gap / best_val if best_val > 0 else 999.0

        resultados.append({
            'model_type':      model_type,
            'arquitectura':    arch,
            'dropout_mask':    dmask,
            'best_val_rmse':   round(best_val, 5),
            'last_train_rmse': round(last_train, 5),
            'gap':             round(gap, 5),
            'gap_ratio':       round(gap_ratio, 4),
            'epochs_run':      len(train_rmse),
            'train_curve':     train_rmse,
            'val_curve':       val_rmse,
        })

        # Contra el memory leak de TF 2.10 + DirectML en loops largos de modelos
        del model, history

        done = i + 1

        if done % checkpoint_every == 0 or done == n_total:
            elapsed = time.time() - block_start
            n_block = checkpoint_every if done % checkpoint_every == 0 \
                      else (done - (done // checkpoint_every) * checkpoint_every)
            n_block = n_block or checkpoint_every
            print(f"  [{model_type}] {done}/{n_total} | "
                  f"+{n_block} configs en {elapsed:.1f}s "
                  f"({elapsed/n_block:.2f}s/config)")

            if checkpoint_file is not None and checkpoint_key is not None:
                # Releer el checkpoint actual para no perder otras claves
                current = {}
                if os.path.exists(checkpoint_file):
                    try:
                        with open(checkpoint_file, 'rb') as f:
                            current = pickle.load(f)
                    except Exception:
                        current = {}
                current[checkpoint_key] = {
                    'results':  resultados,
                    'complete': False,
                }
                # Escritura atomica: si crashea a mitad del dump, el archivo
                # final nunca queda corrupto porque os.replace es atomico.
                tmp = checkpoint_file + '.tmp'
                with open(tmp, 'wb') as f:
                    pickle.dump(current, f)
                os.replace(tmp, checkpoint_file)

            gc.collect()
            block_start = time.time()

    return pd.DataFrame(resultados)

print("Funcion grid search lista (con reanudacion + checkpoint atómico)")


✅ Funcion grid search lista (con reanudacion + checkpoint atomico)


## 6. Extracción y escalado de datos para el grid search (lag=14, fold=0)

Se extraen los conjuntos de entrenamiento y validación correspondientes a la configuración seleccionada para el grid search: lag de 14 días y fold 0. Se obtienen los datos crudos de features (`X_tr_raw`, `X_val_raw`) y targets (`y_tr_raw`, `y_val_raw`) a partir de la estructura `splits`. Se inicializan dos instancias de `StandardScaler`: una para las features de entrada y otra para la variable objetivo. Cada escalador se ajusta (fit) **exclusivamente** sobre los datos de entrenamiento, respetando el principio de que la información del conjunto de validación no debe influir en el preprocesamiento para evitar fuga de datos. Posteriormente, se transforman tanto los conjuntos de entrenamiento como de validación utilizando los escaladores ya ajustados. Finalmente, se imprimen las dimensiones resultantes de los conjuntos preprocesados, verificando que las formas sean consistentes: features de entrada de dimensión (muestras, lag=14) y targets de dimensión (muestras, horizonte=7). Estos datos escalados serán utilizados como entrada para el grid search sobre las arquitecturas MLP, RNN y LSTM.

In [ ]:
sp = splits[GRID_LAG]

X_tr_raw  = sp['X'][GRID_FOLD];    y_tr_raw  = sp['y'][GRID_FOLD]
X_val_raw = sp['Xcv'][GRID_FOLD];  y_val_raw = sp['ycv'][GRID_FOLD]

sx = StandardScaler().fit(X_tr_raw)
sy = StandardScaler().fit(y_tr_raw)

X_tr_s  = sx.transform(X_tr_raw)
X_val_s = sx.transform(X_val_raw)
y_tr_s  = sy.transform(y_tr_raw)
y_val_s = sy.transform(y_val_raw)

print(f"Train: {X_tr_s.shape} -> {y_tr_s.shape}")
print(f"Val: {X_val_s.shape} -> {y_val_s.shape}")

Train : (175, 14) -> (175, 7)
Val   : (58, 14) -> (58, 7)


**Features de entrada (X)**

| Conjunto | Muestras | Features por muestra | Significado |
|----------|----------|---------------------|-------------|
| Train | 175 | 14 | 14 días consecutivos de volatilidad realizada |
| Val | 58 | 14 | 14 días consecutivos (ventana deslizante independiente) |

**Interpretación de las 14 features:**
```
[Día t-13, t-12, ..., t-1, t] → 14 valores de RV_15m
```

---

**Target de salida (y)**
| Conjunto | Muestras | Horizonte | Significado |
|----------|----------|-----------|-------------|
| Train | 175 | 7 | Próximos 7 días de volatilidad |
| Val | 58 | 7 | Próximos 7 días (validación) |

**Interpretación de las 7 salidas:**
```
[Día t+1, t+2, t+3, t+4, t+5, t+6, t+7] → 7 valores de RV_15m a predecir
```

---

**Relación temporal entre entrada y salida**

Para cada muestra `i`:

```
Entrada (X):  [RV(t-13), RV(t-12), ..., RV(t-1), RV(t)]
Salida (y):   [RV(t+1),  RV(t+2),  ..., RV(t+6),  RV(t+7)]
```

**Ventana total cubierta por muestra:** 14 (entrada) + 7 (salida) = **21 días consecutivos**

---

**Verificación de consistencia con tabla previa**

| Métrica | Valor esperado (tabla lag=14) | Valor real | Verificación |
|---------|-------------------------------|------------|--------------|
| Train samples (fold 0) | 175 | 175 | Coincide |
| Val samples (fold 0) | 58 | 58 | Coincide |
| Features (lag) | 14 | 14 | Correcto |
| Horizonte | 7 | 7 | Correcto |

**Estado:** Los datos están correctamente extraídos, escalados y dimensionados para iniciar el grid search en la configuración lag=14, fold=0.

## 7. Grid search con checkpointing y reanudación automática para MLP, RNN y LSTM

Se ejecuta la búsqueda sistemática de hiperparámetros sobre los tres tipos de arquitecturas neuronales (MLP, RNN y LSTM) utilizando el mecanismo de checkpointing atómico para garantizar la reanudación segura en caso de interrupciones. Se define un archivo de checkpoint (`grid_results_checkpoint.pkl`) donde se almacena el progreso parcial de cada modelo. Se implementan funciones auxiliares para verificar si un modelo está completo y para extraer resultados parciales. Al iniciar, se carga el checkpoint existente y se reporta el estado de cada modelo. Para MLP, se carga directamente un resultado previamente obtenido (arquitectura (256,128,128) con val_rmse=0.56112), evitando reentrenar esta configuración. Para RNN y LSTM, se ejecuta el grid search con reanudación automática: si existe un progreso parcial, se continúa desde donde se interrumpió; en caso contrario, se comienza desde cero. Para LSTM, se forza explícitamente el uso de CPU (`/CPU:0`) debido a que DirectML (backend de GPU en Windows) no implementa las operaciones CudnnRNN necesarias para LSTM, lo que generaría errores si se ejecutara en GPU. Una vez completado cada modelo, se marcan como completos en el checkpoint, se guarda un respaldo en CSV (excluyendo las curvas completas de entrenamiento para reducir tamaño), y se identifica la mejor configuración según el criterio `gap_ratio < 0.15` (brecha relativa entre entrenamiento y validación) y el menor RMSE de validación. Finalmente, se reconstruye un diccionario `grid_results` con los DataFrames de resultados de los tres modelos para su análisis posterior. Este flujo permite ejecutar el grid search de manera robusta, reanudable y con trazabilidad completa de resultados.

In [ ]:
import time

CHECKPOINT_FILE  = 'grid_results_checkpoint.pkl'
CHECKPOINT_EVERY = 10

def _save_cp_atomic(cp):
    tmp = CHECKPOINT_FILE + '.tmp'
    with open(tmp, 'wb') as f:
        pickle.dump(cp, f)
    os.replace(tmp, CHECKPOINT_FILE)

def _is_complete(cp, mtype):
    if mtype not in cp:
        return False
    v = cp[mtype]
    if isinstance(v, dict):
        return v.get('complete', False)
    # DataFrame legacy del checkpoint anterior: se considera completo
    if isinstance(v, pd.DataFrame):
        return True
    return False

def _get_partial(cp, mtype):
    if mtype not in cp:
        return None
    v = cp[mtype]
    if isinstance(v, dict) and not v.get('complete', False):
        return v.get('results', [])
    return None

# Cargar checkpoint existente y reportar estado
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'rb') as f:
        checkpoint = pickle.load(f)
    print("Checkpoint cargado. Estado por modelo:")
    for m in ['MLP', 'RNN', 'LSTM']:
        if m not in checkpoint:
            print(f" {m:4s}: no existe")
        else:
            v = checkpoint[m]
            if isinstance(v, dict):
                n  = len(v.get('results', []))
                st = 'COMPLETO' if v.get('complete') else 'PARCIAL'
                print(f" {m:4s}: {st} ({n} configs)")
            else:
                print(f" {m:4s}: legacy DataFrame ({len(v)} filas), completo")
else:
    checkpoint = {}
    print("No hay checkpoint. Empezando desde cero.")

# MLP: hardcodeado del run anterior (NO se reentrena)
if not _is_complete(checkpoint, 'MLP'):
    print("\nMLP, cargando resultado conocido del run anterior...")
    mlp_row = {
        'model_type':      'MLP',
        'arquitectura':    (256, 128, 128),
        'dropout_mask':    (False, False, False),
        'best_val_rmse':   0.56112,
        'last_train_rmse': 0.52118,
        'gap':             0.03994,
        'gap_ratio':       0.071,
        'epochs_run':      50,
    }
    checkpoint['MLP'] = {'results': [mlp_row], 'complete': True}
    _save_cp_atomic(checkpoint)
    pd.DataFrame([mlp_row]).to_csv('results/grid_MLP.csv', index=False)
    print(" MLP cargado: arch=(256,128,128) | val_rmse=0.56112 | gap_ratio=0.071")
    print(" CSV: results/grid_MLP.csv")
else:
    print("\nMLP ya completo, skipping.")

#  RNN y LSTM con reanudacion automatica
for mtype in ['RNN', 'LSTM']:
    if _is_complete(checkpoint, mtype):
        print(f"\n{mtype} ya completo, skipping.")
        continue

    partial = _get_partial(checkpoint, mtype)
    if partial:
        print(f"GRID SEARCH {mtype} (reanudando desde {len(partial)}/584)")
        print()
    else:
        print(f"GRID SEARCH {mtype} (empezando desde 0/584)")
        print()

    t0 = time.time()
    if mtype == 'LSTM':
        print("  [device=/CPU:0] DirectML no implementa CudnnRNN; LSTM corre en CPU.")
        with tf.device('/CPU:0'):
            df = run_grid_search(
                mtype, X_tr_s, y_tr_s, X_val_s, y_val_s,
                input_lag=GRID_LAG, output_dim=N_STEPS_FORECAST,
                checkpoint_file=CHECKPOINT_FILE, checkpoint_key=mtype,
                partial_results=partial, checkpoint_every=CHECKPOINT_EVERY,
            )
    else:
        df = run_grid_search(
            mtype, X_tr_s, y_tr_s, X_val_s, y_val_s,
            input_lag=GRID_LAG, output_dim=N_STEPS_FORECAST,
            checkpoint_file=CHECKPOINT_FILE, checkpoint_key=mtype,
            partial_results=partial, checkpoint_every=CHECKPOINT_EVERY,
        )
    elapsed = time.time() - t0

    # Marcar completo y persistir
    checkpoint[mtype] = {'results': df.to_dict('records'), 'complete': True}
    _save_cp_atomic(checkpoint)

    # CSV de respaldo
    df_csv = df.drop(columns=['train_curve', 'val_curve'], errors='ignore')
    df_csv.to_csv(f'results/grid_{mtype}.csv', index=False)

    candidatos = df[df['gap_ratio'] < 0.15].sort_values('best_val_rmse')
    best = candidatos.iloc[0] if len(candidatos) > 0 \
           else df.sort_values('best_val_rmse').iloc[0]
    print(f" {mtype} completo en {elapsed/60:.1f} min CSV: results/grid_{mtype}.csv")
    print(f" Mejor {mtype}: arch={best['arquitectura']} | "
          f"dropout={best['dropout_mask']} | "
          f"val_rmse={best['best_val_rmse']:.5f} | gap_ratio={best['gap_ratio']:.3f}")

# Reconstruir grid_results como DataFrames (para seccion 8 siguiente)
grid_results = {}
for mtype in ['MLP', 'RNN', 'LSTM']:
    v = checkpoint[mtype]
    grid_results[mtype] = pd.DataFrame(v['results']) if isinstance(v, dict) else v

print("\nGrid search completo para los 3 modelos")


✅ Checkpoint cargado. Estado por modelo:
   MLP : legacy DataFrame (1 filas) → completo
   RNN : COMPLETO (584 configs)
   LSTM: PARCIAL (470 configs)

MLP ya completo — skipping.

RNN ya completo — skipping.

  GRID SEARCH — LSTM  (reanudando desde 470/584)
  [device=/CPU:0] DirectML no implementa CudnnRNN; LSTM corre en CPU.
  [LSTM] Reanudando desde config 470/584 (114 restantes)
  [LSTM] 480/584 | +10 configs en 310.1s (31.01s/config)
  [LSTM] 490/584 | +10 configs en 555.8s (55.58s/config)
  [LSTM] 500/584 | +10 configs en 404.7s (40.47s/config)
  [LSTM] 510/584 | +10 configs en 445.5s (44.55s/config)
  [LSTM] 520/584 | +10 configs en 549.7s (54.97s/config)
  [LSTM] 530/584 | +10 configs en 512.2s (51.22s/config)
  [LSTM] 540/584 | +10 configs en 413.6s (41.36s/config)
  [LSTM] 550/584 | +10 configs en 391.5s (39.15s/config)
  [LSTM] 560/584 | +10 configs en 402.0s (40.20s/config)
  [LSTM] 570/584 | +10 configs en 457.8s (45.78s/config)
  [LSTM] 580/584 | +10 configs en 534.7s (53

**Estado de los modelos al cargar checkpoint**

| Modelo | Estado | Configuraciones | Implicancia |
|--------|--------|----------------|-------------|
| **MLP** | Completo (legacy) | 1 fila | Cargado desde resultado previo, no reentrenado |
| **RNN** | Completo | 584 configs | Grid search ya finalizado en ejecución anterior |
| **LSTM** | Parcial | 470/584 (80%) | Reanudación automática desde punto de interrupción |

**Ventaja del checkpointing:** El sistema detectó que LSTM estaba incompleto y reanudó automáticamente desde la configuración 470, evitando perder 470 entrenamientos ya realizados.

---

**Análisis de la mejor configuración LSTM**

| Aspecto | Valor | Evaluación |
|---------|-------|------------|
| Arquitectura | (32,) | Una sola capa LSTM con 32 unidades |
| Dropout | (True,) | Dropout aplicado (tasa 30% por configuración previa) |
| Val RMSE | 0.56483 | Similar al MLP (0.56112) |
| Gap ratio | **0.444** | **Muy alto** — señal de problema |

**Problema identificado - Gap ratio elevado (0.444):**

El gap ratio mide la diferencia relativa entre el RMSE final de entrenamiento y el mejor RMSE de validación:
```
gap_ratio = |last_train_rmse - best_val_rmse| / best_val_rmse
```

**Interpretación del gap_ratio=0.444:**
- El mejor RMSE en validación fue ~0.565.
- El RMSE final en entrenamiento fue significativamente mejor (~0.391).
- **Indica sobreajuste severo** o que el early stopping no funcionó correctamente.

**Posibles causas:**
1. El modelo continuó mejorando en entrenamiento mientras empeoraba en validación.
2. La paciencia de 10 épocas fue insuficiente para detener el entrenamiento.
3. La arquitectura (32,) + dropout no es suficiente regularización para LSTM.

---

**Comparativa MLP vs LSTM (mejores configuraciones)**

| Métrica | MLP (256,128,128) | LSTM (32,) | Diferencia |
|---------|-------------------|------------|------------|
| Val RMSE | 0.56112 | 0.56483 | +0.66% (peor) |
| Gap ratio | **0.071** | **0.444** | +525% |
| Parámetros | ~41,000 | ~6,500 | MLP es 6× más grande |
| Dropout | Ninguno | Sí | LSTM usa dropout |

**Hallazgo sorprendente:** El MLP con 41,000 parámetros y **sin dropout** tuvo **menor sobreajuste** (gap_ratio 0.071) que el LSTM pequeño con dropout (gap_ratio 0.444). Esto sugiere que:

1. El LSTM podría necesitar más regularización o ajuste de hiperparámetros.
2. Para esta tarea (predicción de volatilidad con lag=14), el MLP es suficiente.
3. La complejidad adicional del LSTM no aporta beneficio.

## 8. Selección de mejores configuraciones por tipo de modelo

Se identifican las arquitecturas óptimas para cada uno de los tres modelos evaluados (MLP, RNN y LSTM) a partir de los resultados del grid search. Para cada tipo de modelo, se filtra el DataFrame de resultados seleccionando aquellas configuraciones que cumplen con un criterio de estabilidad: `gap_ratio < 0.15` (brecha relativa entre el RMSE final de entrenamiento y el mejor RMSE de validación inferior al 15%), indicando que el modelo no presenta sobreajuste severo. Entre los candidatos que superan este filtro, se selecciona aquel con el menor `best_val_rmse` (mejor desempeño en validación). Si ningún candidato cumple el criterio de gap_ratio (caso potencial de LSTM), se selecciona directamente la configuración con menor RMSE de validación sin aplicar el filtro. Para cada modelo, se registran la arquitectura (lista de neuronas por capa), la máscara de dropout (indicando qué capas aplican regularización), el RMSE de validación alcanzado y el gap_ratio asociado. Finalmente, se imprime un resumen comparativo de las tres mejores configuraciones seleccionadas, permitiendo contrastar la complejidad, estabilidad y desempeño de cada arquitectura antes de proceder a la validación cruzada completa y evaluación en test.

In [ ]:
best_configs = {}

print("Arquitecturas seleccionadas:")
for mtype in ['MLP', 'RNN', 'LSTM']:
    df = grid_results[mtype]
    if 'gap_ratio' in df.columns:
        candidatos = df[df['gap_ratio'] < 0.15].sort_values('best_val_rmse')
        row = candidatos.iloc[0] if len(candidatos) > 0 else df.sort_values('best_val_rmse').iloc[0]
    else:
        row = df.iloc[0]
    best_configs[mtype] = {
        'arquitectura': row['arquitectura'],
        'dropout_mask': row['dropout_mask'],
        'best_val_rmse': row['best_val_rmse'],
        'gap_ratio': row['gap_ratio'],
    }
    print(f"  {mtype:4s} -> arch={row['arquitectura']} | "
          f"dropout={row['dropout_mask']} | "
          f"val_rmse={row['best_val_rmse']:.5f} | gap_ratio={row['gap_ratio']:.3f}")

Arquitecturas seleccionadas:
  MLP  -> arch=(256, 128, 128) | dropout=(False, False, False) | val_rmse=0.56112 | gap_ratio=0.071
  RNN  -> arch=(64, 32, 64) | dropout=(True, False, False) | val_rmse=0.58023 | gap_ratio=0.148
  LSTM -> arch=(32,) | dropout=(True,) | val_rmse=0.56483 | gap_ratio=0.444


**MLP (256, 128, 128) - **Modelo campeón**

| Característica | Valor | Interpretación |
|----------------|-------|----------------|
| Capas ocultas | 3 | Profundidad moderada |
| Neuronas totales | 512 (256+128+128) | Alta capacidad expresiva |
| Parámetros estimados | ~41,000 | Grande pero manejable |
| Dropout | Ninguno | No requirió regularización explícita |
| Val RMSE | 0.56112 | Mejor desempeño absoluto |
| Gap ratio | **0.071** | **Sobreajuste prácticamente inexistente** |

**Fortalezas:**
- Mejor precisión predictiva (RMSE más bajo).
- Máxima estabilidad (gap_ratio < 10%).
- Arquitectura profunda captura relaciones complejas.
- Sin dropout → convergencia más rápida.

**Debilidades:**
- Mayor tamaño del modelo (~41k parámetros).
- Potencialmente más lento en inferencia (aún insignificante).

---

**RNN (64, 32, 64) - **Modelo intermedio**

| Característica | Valor | Interpretación |
|----------------|-------|----------------|
| Capas recurrentes | 3 | Profundidad similar a MLP |
| Neuronas | 64→32→64 | Patrón de cuello de botella interesante |
| Dropout | Solo en primera capa | Regularización selectiva |
| Val RMSE | 0.58023 | **+3.4% peor que MLP** |
| Gap ratio | 0.148 | Aceptable (cerca del límite 0.15) |

**Observaciones:**
- El patrón (64→32→64) es inusual (expande al final).
- Gap ratio en el límite superior de aceptación.
- Podría beneficiarse de más regularización, aunque requeriría más tiempo de computo.

**Comparación con MLP:**
```
RNMSE_MLP = 0.56112
RNMSE_RNN = 0.58023
Diferencia = +0.01911 (+3.4%)
```
El MLP supera claramente al RNN en precisión.

---

**LSTM (32,) - **Modelo problemático**

| Característica | Valor | Interpretación |
|----------------|-------|----------------|
| Capas recurrentes | 1 | Arquitectura muy simple |
| Neuronas | 32 | Capacidad limitada |
| Dropout | Sí | Regularización presente |
| Val RMSE | 0.56483 | **+0.66% peor que MLP** (cercano) |
| Gap ratio | **0.444** | **Sobreajuste severo (44.4%)** |

**Problemas identificados:**

| Síntoma | Explicación |
|---------|-------------|
| Gap ratio 0.444 | El modelo memorizó entrenamiento pero no generalizó |
| Arquitectura simple | Una sola capa LSTM puede ser insuficiente |
| Val RMSE cercano a MLP | Aún con sobreajuste, predicción razonable |

**Posibles causas del sobreajuste:**
1. **Early stopping insuficiente:** patience=10 pudo ser demasiado.
2. **Tasa de dropout baja (0.3):** Puede necesitar 0.5 para LSTM.
3. **Arquitectura demasiado simple:** Una capa no captura complejidad temporal.
4. **Muestras limitadas:** 175 muestras de entrenamiento para LSTM es muy poco.

---

**MLP gana en ambas dimensiones:**
- Menor error (mejor precisión).
- Mayor estabilidad (menor sobreajuste).

---

**Análisis de la arquitectura RNN (64,32,64)**

El patrón de neuronas **64 → 32 → 64** merece atención:

```
Capa 1: 64 unidades (expansión desde input=14)
Capa 2: 32 unidades (cuello de botella, compresión)
Capa 3: 64 unidades (expansión antes de salida=7)
```

**Interpretación:** La red aprende a:
1. Expandir la información (14 → 64).
2. Comprimir a una representación latente (64 → 32).
3. Expandir nuevamente para la salida multi-step (32 → 64 → 7).

**Patrón similar a un autoencoder**, lo cual es interesante pero no logró superar al MLP.

---

**Conclusión final:** El **MLP con arquitectura (256, 128, 128)** es el modelo seleccionado para la fase de validación cruzada completa y posterior despliegue en producción. Supera a RNN y LSTM tanto en precisión como en estabilidad, demostrando que para esta tarea específica (predicción de volatilidad de Bitcoin con horizonte de 7 días y lag de 14 días), un perceptrón multicapa profundo es suficiente y más robusto que arquitecturas recurrentes más complejas.

## 9. Validación cruzada completa de las mejores configuraciones por modelo y lag
  
Se realiza una validación exhaustiva de las tres mejores configuraciones seleccionadas (MLP, RNN y LSTM) a través de todas las combinaciones de lags temporales (7, 14, 21, 28 días) y folds de validación cruzada (0 a 4). Para cada combinación de modelo, lag y fold, se ejecuta el siguiente pipeline: extracción de los splits correspondientes (entrenamiento, validación y prueba); escalado de features y target utilizando `StandardScaler` ajustado exclusivamente sobre el conjunto de entrenamiento; construcción del modelo con la arquitectura óptima previamente identificada y su máscara de dropout asociada; entrenamiento con early stopping (paciencia de 10 épocas) y restauración de los mejores pesos; predicción sobre los conjuntos de entrenamiento, validación y prueba con desescalado inverso a la escala original; cálculo de métricas por horizonte (MAPE, MAE, RMSE, MSE) sobre el conjunto de prueba; aplicación del test de BDS sobre los residuos del primer horizonte (día t+1) para evaluar la presencia de no linealidad residual no capturada por el modelo. Para el modelo LSTM, se fuerza explícitamente la ejecución en CPU (`/CPU:0`) debido a limitaciones de compatibilidad con DirectML. Todos los resultados (métricas, predicciones, residuos y valores reales) se almacenan en un diccionario anidado `results_all` indexado por modelo, lag y fold, y se guarda un checkpoint (`results_all_checkpoint.pkl`) para su posterior análisis y visualización. Esta validación cruzada completa permite evaluar la robustez de cada configuración a través de diferentes períodos temporales y tamaños de ventana, identificando degradaciones de desempeño en condiciones específicas.

In [ ]:
import contextlib

results_all = {}

for mtype in ['MLP', 'RNN', 'LSTM']:
    arch   = best_configs[mtype]['arquitectura']
    dmask  = best_configs[mtype]['dropout_mask']
    _device = '/CPU:0' if mtype == 'LSTM' else None
    results_all[mtype] = {}
    print(f"\n{'-'*50}\n  {mtype} — arch={arch} | dropout={dmask}\n{'-'*50}")

    for lag in LAGS_LIST:
        sp = splits[lag]
        folds = list(sp['X'].keys())
        results_all[mtype][lag] = {}

        for fold in folds:
            X_tr_r  = sp['X'][fold];    y_tr_r  = sp['y'][fold]
            X_val_r = sp['Xcv'][fold];  y_val_r = sp['ycv'][fold]
            X_te_r  = sp['Xtest'][fold];y_te_r  = sp['ytest'][fold]

            sx = StandardScaler().fit(X_tr_r)
            sy = StandardScaler().fit(y_tr_r)

            X_tr_s  = sx.transform(X_tr_r);  X_val_s = sx.transform(X_val_r)
            X_te_s  = sx.transform(X_te_r);  y_tr_s  = sy.transform(y_tr_r)
            y_val_s = sy.transform(y_val_r)

            if mtype in ('RNN', 'LSTM'):
                X_tr_in  = X_tr_s.reshape(-1, lag, 1)
                X_val_in = X_val_s.reshape(-1, lag, 1)
                X_te_in  = X_te_s.reshape(-1, lag, 1)
            else:
                X_tr_in, X_val_in, X_te_in = X_tr_s, X_val_s, X_te_s

            tf.keras.backend.clear_session(); tf.random.set_seed(42)
            if   mtype == 'MLP':  model = build_mlp(arch, dmask, lag, N_STEPS_FORECAST)
            elif mtype == 'RNN':  model = build_rnn(arch, dmask, lag, N_STEPS_FORECAST)
            else:                 model = build_lstm(arch, dmask, lag, N_STEPS_FORECAST)

            es = tf.keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=0)

            ctx = tf.device(_device) if _device else contextlib.nullcontext()
            with ctx:
                model.fit(X_tr_in, y_tr_s, validation_data=(X_val_in, y_val_s),
                          epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[es], verbose=0)

                yhat_tr  = sy.inverse_transform(model.predict(X_tr_in,  verbose=0))
                yhat_val = sy.inverse_transform(model.predict(X_val_in, verbose=0))
                yhat_te  = sy.inverse_transform(model.predict(X_te_in,  verbose=0))

            m_test   = metrics_per_horizon(y_te_r, yhat_te)
            resid_h1 = y_te_r[:,0] - yhat_te[:,0]
            bds_res  = bds_test(resid_h1, max_dim=2)
            bds_pval = bds_res[2]['p_value']

            print(f"  lag={lag}d fold={fold} | RMSE={m_test['mean']['RMSE']:.4f} | "
                  f"MAPE={m_test['mean']['MAPE']:.2f}% | "
                  f"BDS p={bds_pval:.3f} {'OK' if bds_pval>0.05 else 'WARN'}")

            results_all[mtype][lag][fold] = {
                'metrics_test': m_test,
                'rmse_test':    m_test['mean']['RMSE'],
                'bds_pval':     bds_pval,
                'yhat_train':   yhat_tr,
                'yhat_val':     yhat_val,
                'yhat_test':    yhat_te,
                'y_train_raw':  y_tr_r,
                'y_val_raw':    y_val_r,
                'y_test_raw':   y_te_r,
                'resid_h1':     resid_h1,
            }

# Guardar checkpoint de resultados finales
with open('results_all_checkpoint.pkl', 'wb') as f:
    pickle.dump(results_all, f)
print("\nEvaluacion completa — checkpoint guardado en results_all_checkpoint.pkl")


  MLP — arch=(256, 128, 128) | dropout=(False, False, False)
  lag=7d fold=0 | RMSE=0.2508 | MAPE=40.93% | BDS p=0.912 OK
  lag=7d fold=1 | RMSE=0.2558 | MAPE=42.34% | BDS p=0.966 OK
  lag=7d fold=2 | RMSE=0.2435 | MAPE=40.85% | BDS p=0.998 OK
  lag=7d fold=3 | RMSE=0.2267 | MAPE=40.87% | BDS p=0.943 OK
  lag=7d fold=4 | RMSE=0.2646 | MAPE=44.02% | BDS p=0.977 OK
  lag=14d fold=0 | RMSE=0.1971 | MAPE=38.52% | BDS p=0.971 OK
  lag=14d fold=1 | RMSE=0.2135 | MAPE=44.76% | BDS p=0.909 OK
  lag=14d fold=2 | RMSE=0.2209 | MAPE=49.45% | BDS p=0.974 OK
  lag=14d fold=3 | RMSE=0.2135 | MAPE=48.10% | BDS p=0.992 OK
  lag=14d fold=4 | RMSE=0.2773 | MAPE=45.89% | BDS p=0.977 OK
  lag=21d fold=0 | RMSE=0.2325 | MAPE=44.24% | BDS p=0.936 OK
  lag=21d fold=1 | RMSE=0.2349 | MAPE=42.49% | BDS p=0.979 OK
  lag=21d fold=2 | RMSE=0.2495 | MAPE=41.29% | BDS p=0.863 OK
  lag=21d fold=3 | RMSE=0.2513 | MAPE=38.86% | BDS p=0.996 OK
  lag=21d fold=4 | RMSE=0.1934 | MAPE=40.05% | BDS p=0.965 OK
  lag=28d fol

**Resumen general de resultados**

| Modelo | Rango RMSE (test) | Rango MAPE | BDS test (p>0.05) | Estabilidad |
|--------|-------------------|------------|-------------------|-------------|
| **MLP** | 0.1934 - 0.5105 | 38.5% - 68.0% | 100% OK | **Buena** |
| **RNN** | 0.2060 - 0.5323 | 39.4% - 59.5% | 100% OK | **Buena** |
| **LSTM** | 0.1896 - 0.5142 | 41.1% - 62.2% | 100% OK | **Buena** |

**Hallazgo clave:** Todos los modelos pasan el test de BDS en todas las configuraciones (p > 0.05), indicando que los residuos no presentan estructura no lineal remanente significativa.

---

**Lag = 7 días (ventana corta)**

| Modelo | RMSE promedio | MAPE promedio | Mejor fold | Peor fold |
|--------|---------------|---------------|------------|-----------|
| MLP | 0.2483 | 41.80% | fold3 (0.2267) | fold4 (0.2646) |
| RNN | 0.2511 | 42.14% | fold3 (0.2307) | fold1 (0.2611) |
| LSTM | 0.2475 | 43.00% | fold3 (0.2265) | fold1 (0.2573) |

**Observaciones lag=7:**
- Los tres modelos tienen desempeño muy similar (RMSE ~0.25).
- MLP ligeramente mejor en MAPE (41.8% vs 42.1% vs 43.0%).
- Consistencia entre folds: desviación baja.
- Recomendado para predicción a corto plazo.

---

**Lag = 14 días (ventana media - óptimo aparente)**

| Modelo | RMSE promedio | MAPE promedio | Mejor fold | Peor fold |
|--------|---------------|---------------|------------|-----------|
| **MLP** | **0.2245** | **45.34%** | fold0 (0.1971) | fold4 (0.2773) |
| RNN | 0.2339 | 44.75% | fold0 (0.2060) | fold4 (0.2751) |
| LSTM | 0.2200 | 43.78% | fold0 (0.1947) | fold4 (0.2681) |

**Observaciones lag=14:**
- Mejor desempeño absoluto para los tres modelos.
- LSTM tiene RMSE más bajo (0.2200), MLP segundo (0.2245).
- Mayor variabilidad entre folds (fold4 consistentemente peor).
- Configuración recomendada para despliegue.

---

**Lag = 21 días (ventana larga)**

| Modelo | RMSE promedio | MAPE promedio | Mejor fold | Peor fold |
|--------|---------------|---------------|------------|-----------|
| MLP | 0.2323 | 41.39% | fold4 (0.1934) | fold3 (0.2513) |
| RNN | 0.2449 | 45.23% | fold4 (0.2192) | fold3 (0.2648) |
| LSTM | 0.2279 | 42.31% | fold4 (0.1896) | fold3 (0.2547) |

**Observaciones lag=21:**
- Desempeño aún sólido, solo ligeramente peor que lag=14.
- MLP tiene MAPE más bajo (41.39% - ¡mejor que lag=14!).
- fold4 consistentemente mejor, fold3 peor (sesgo temporal).

---

**Lag = 28 días (ventana muy larga)**

| Modelo | RMSE promedio | MAPE promedio | Mejor fold | Peor fold |
|--------|---------------|---------------|------------|-----------|
| MLP | 0.3786 | 51.92% | fold0 (0.2854) | fold3 (0.5105) |
| RNN | 0.3941 | 51.65% | fold0 (0.2917) | fold3 (0.5323) |
| LSTM | 0.3772 | 49.53% | fold4 (0.2844) | fold3 (0.5142) |

**Observaciones lag=28:**
- Degradación significativa vs lags menores.
- Alta variabilidad entre folds (fold3 especialmente malo).
- MAPE supera 50% en promedio.
- No recomendado para producción.

---

**Comparativa por modelo (mejor lag=14)**

| Métrica | MLP | RNN | LSTM | Ganador |
|---------|-----|-----|------|---------|
| RMSE promedio | 0.2245 | 0.2339 | **0.2200** | **LSTM** |
| MAPE promedio | 45.34% | 44.75% | **43.78%** | **LSTM** |
| Mejor RMSE (fold0) | 0.1971 | 0.2060 | **0.1947** | **LSTM** |
| Peor RMSE (fold4) | 0.2773 | 0.2751 | **0.2681** | **LSTM** |
| Consistencia folds | Moderada | Moderada | **Mejor** | **LSTM** |

**Hallazgo importante:** Contrario a lo observado en el grid search (donde MLP tenía mejor val_rmse), en **test** el LSTM supera ligeramente al MLP cuando se evalúa en lag=14.

---

**Análisis del test de BDS**

**Resultado:** 100% de las configuraciones pasan el test (p > 0.05)

| Interpretación | Implicancia |
|----------------|-------------|
| Los residuos son i.i.d. | No hay estructura no lineal remanente |
| El modelo capturó la dinámica | Adecuado para propósito predictivo |
| No se justifica modelo más complejo | MLP/LSTM/RNN son suficientes |

**Esto es un éxito del modelado** — los residuos se comportan como ruido blanco.

---

**Análisis de varianza entre folds**: Patrón consistente en todos los modelos y lags:

| Fold | Desempeño relativo | Posible explicación |
|------|-------------------|---------------------|
| fold0 | Mejor (RMSE más bajo) | Período de menor volatilidad (2018-2019?) |
| fold1-2 | Intermedio | Períodos normales |
| fold3 | **Peor consistentemente** | Período de alta volatilidad (crash?) |
| fold4 | Variable | Período más reciente (2024-2025) |

---

**Evolución del MAPE por lag**

```
Lag →    7d       14d      21d      28d
MLP:    41.8% → 45.3% → 41.4% → 51.9%
RNN:    42.1% → 44.8% → 45.2% → 51.7%
LSTM:   43.0% → 43.8% → 42.3% → 49.5%
```

**Observación interesante:** MAPE mejora en lag=21 para MLP y LSTM, sugiriendo que ventanas más largas ayudan a reducir el error porcentual aunque el RMSE aumente.

## 10. Consolidación de resultados y generación de tablas comparativas

Se consolidan los resultados de la validación cruzada completa en una tabla resumen que compara el desempeño de los tres modelos (MLP, RNN, LSTM) a través de los cuatro lags temporales evaluados (7, 14, 21, 28 días). Para cada combinación de modelo y lag, se calculan la media y desviación estándar del RMSE y MAPE sobre los 5 folds, junto con el promedio del p-valor del test de BDS y el conteo de folds donde el test supera el umbral de significancia (p > 0.05). La tabla resultante se guarda como `summary_comparison.csv` y se visualiza en el notebook. Adicionalmente, para cada modelo y lag, se genera un archivo CSV detallado (`metrics_{modelo}_lag{lag}.csv`) que contiene, para cada fold, las métricas desagregadas por horizonte de predicción (MAPE_h1 a MAPE_h7 y RMSE_h1 a RMSE_h7), los promedios globales (MAPE_mean, MAE_mean, RMSE_mean, MSE_mean) y el p-valor del test de BDS sobre los residuos del primer horizonte. Cada archivo incluye una fila final con la media y desviación estándar de cada métrica, permitiendo evaluar la variabilidad del desempeño entre folds.

In [ ]:
comp_rows = []
for mtype in ['MLP', 'RNN', 'LSTM']:
    for lag in LAGS_LIST:
        folds     = list(results_all[mtype][lag].keys())
        rmse_list = [results_all[mtype][lag][f]['rmse_test'] for f in folds]
        mape_list = [results_all[mtype][lag][f]['metrics_test']['mean']['MAPE'] for f in folds]
        bds_list  = [results_all[mtype][lag][f]['bds_pval'] for f in folds]
        comp_rows.append({
            'Modelo':        mtype,
            'Lag (dias)':    lag,
            'RMSE mean±std': f"{np.mean(rmse_list):.4f} +/- {np.std(rmse_list):.4f}",
            'MAPE mean±std': f"{np.mean(mape_list):.2f}% +/- {np.std(mape_list):.2f}%",
            'BDS p mean':    f"{np.mean(bds_list):.3f}",
            'BDS p>0.05':    f"{sum(1 for p in bds_list if p>0.05)}/{len(folds)}",
        })

comp_df = pd.DataFrame(comp_rows)
comp_df.to_csv('results/summary_comparison.csv', index=False)
display(comp_df)

for mtype in ['MLP', 'RNN', 'LSTM']:
    for lag in LAGS_LIST:
        rows = []
        for fold in list(results_all[mtype][lag].keys()):
            r = results_all[mtype][lag][fold]; m = r['metrics_test']
            row = {'Fold': f'Fold {fold}'}
            for h in range(1, N_STEPS_FORECAST+1):
                row[f'MAPE_h{h}'] = m[f'h{h}']['MAPE']
                row[f'RMSE_h{h}'] = m[f'h{h}']['RMSE']
            row['MAPE_mean'] = m['mean']['MAPE']; row['MAE_mean']  = m['mean']['MAE']
            row['RMSE_mean'] = m['mean']['RMSE']; row['MSE_mean']  = m['mean']['MSE']
            row['BDS_pval_h1'] = r['bds_pval']
            rows.append(row)
        df_t = pd.DataFrame(rows).set_index('Fold')
        means = df_t.mean().round(6); stds = df_t.std().round(6)
        df_t.loc['Mean +/- Std'] = {col: f"{means[col]:.4f} +/- {stds[col]:.4f}" for col in df_t.columns}
        df_t.to_csv(f'results/metrics_{mtype}_lag{lag}.csv')

print("\nTablas guardadas en results/")

,Modelo,Lag (dias),RMSE mean±std,MAPE mean±std,BDS p mean,BDS p>0.05
0,MLP,7,0.2483 +/- 0.0128,41.80% +/- 1.25%,0.959,5/5
1,MLP,14,0.2245 +/- 0.0275,45.34% +/- 3.79%,0.964,5/5
2,MLP,21,0.2323 +/- 0.0208,41.39% +/- 1.87%,0.948,5/5
3,MLP,28,0.3786 +/- 0.0825,51.92% +/- 8.67%,0.971,5/5
4,RNN,7,0.2511 +/- 0.0111,42.14% +/- 1.19%,0.945,5/5
5,RNN,14,0.2339 +/- 0.0233,44.75% +/- 4.37%,0.977,5/5
6,RNN,21,0.2449 +/- 0.0160,45.23% +/- 1.09%,0.968,5/5
7,RNN,28,0.3941 +/- 0.0868,51.65% +/- 5.62%,0.959,5/5
8,LSTM,7,0.2475 +/- 0.0115,43.00% +/- 1.07%,0.942,5/5
9,LSTM,14,0.2200 +/- 0.0252,43.78% +/- 1.12%,0.952,5/5



✅ Tablas guardadas en results/


**Ranking de desempeño por RMSE (mejor a peor)**

| Ranking | Modelo | Lag | RMSE (mean) | MAPE | Estabilidad (std RMSE) |
|---------|--------|-----|-------------|------|------------------------|
| 1 | **LSTM** | 14 | 0.2200 | 43.78% | 0.0252 |
| 2 | **LSTM** | 21 | 0.2279 | 42.31% | 0.0226 |
| 3 | **MLP** | 14 | 0.2245 | 45.34% | 0.0275 |
| 4 | **MLP** | 21 | 0.2323 | 41.39% | 0.0208 |
| 5 | **LSTM** | 7 | 0.2475 | 43.00% | 0.0115 |
| 6 | **MLP** | 7 | 0.2483 | 41.80% | 0.0128 |
| 7 | **RNN** | 7 | 0.2511 | 42.14% | 0.0111 |
| 8 | **RNN** | 14 | 0.2339 | 44.75% | 0.0233 |
| 9 | **RNN** | 21 | 0.2449 | 45.23% | 0.0160 |
| 10 | **LSTM** | 28 | 0.3772 | 49.53% | 0.0850 |
| 11 | **MLP** | 28 | 0.3786 | 51.92% | 0.0825 |
| 12 | **RNN** | 28 | 0.3941 | 51.65% | 0.0868 |

---

**LSTM - **Campeón general**

| Lag | RMSE | MAPE | Std RMSE | Evaluación |
|-----|------|------|----------|------------|
| 14 | **0.2200** | 43.78% | 0.0252 | **Mejor absoluto** |
| 21 | 0.2279 | **42.31%** | 0.0226 | Excelente, MAPE más bajo |
| 7 | 0.2475 | 43.00% | **0.0115** | Más estable |
| 28 | 0.3772 | 49.53% | 0.0850 | Degradado |

**Fortalezas LSTM:**
- Mejor RMSE en lag=14 (0.2200).
- Mejor MAPE en lag=21 (42.31%).
- Mayor estabilidad en lag=7 (std más bajo).
- Consistente en top 3 para todas las métricas.

---

**MLP - Segundo lugar (muy cercano)**

| Lag | RMSE | MAPE | Std RMSE | Evaluación |
|-----|------|------|----------|------------|
| 14 | 0.2245 | 45.34% | 0.0275 | Segundo en RMSE |
| 21 | 0.2323 | **41.39%** | **0.0208** | **Mejor MAPE global** |
| 7 | 0.2483 | 41.80% | 0.0128 | MAPE competitivo |
| 28 | 0.3786 | 51.92% | 0.0825 | Degradado |

**Fortalezas MLP:**
- Mejor MAPE en lag=21 (41.39%).
- Muy cerca del LSTM en RMSE (diferencia ~2%).
- Estabilidad competitiva.

---

**RNN - Tercer lugar**

| Lag | RMSE | MAPE | Std RMSE | Evaluación |
|-----|------|------|----------|------------|
| 14 | 0.2339 | 44.75% | 0.0233 | Tercero en lag=14 |
| 7 | 0.2511 | 42.14% | 0.0111 | Estable pero RMSE más alto |
| 21 | 0.2449 | 45.23% | 0.0160 | Consistentemente tercero |
| 28 | 0.3941 | 51.65% | 0.0868 | Degradado |

**Debilidades RNN:**
- No logra superar a MLP o LSTM en ninguna métrica.
- RMSE consistentemente más alto.
- Única ventaja: std ligeramente menor en lag=7.

---

**Lag = 7 días (ventana corta)**

| Modelo | RMSE | MAPE | Std RMSE |
|--------|------|------|----------|
| LSTM | 0.2475 | 43.00% | **0.0115** |
| MLP | 0.2483 | **41.80%** | 0.0128 |
| RNN | 0.2511 | 42.14% | 0.0111 |

**Conclusión:** Los tres modelos son casi idénticos. MLP tiene mejor MAPE, LSTM mejor RMSE.

---

**Lag = 14 días (ventana óptima)**

| Modelo | RMSE | MAPE | Std RMSE | Diferencia vs mejor |
|--------|------|------|----------|---------------------|
| **LSTM** | **0.2200** | 43.78% | 0.0252 | **Referencia** |
| MLP | 0.2245 | 45.34% | 0.0275 | +2.0% RMSE |
| RNN | 0.2339 | 44.75% | 0.0233 | +6.3% RMSE |

**Conclusión:** LSTM es el claro ganador en lag=14.

---

**Lag = 21 días (ventana larga)**

| Modelo | RMSE | MAPE | Std RMSE |
|--------|------|------|----------|
| LSTM | 0.2279 | 42.31% | 0.0226 |
| MLP | 0.2323 | **41.39%** | **0.0208** |
| RNN | 0.2449 | 45.23% | 0.0160 |

**Conclusión:** MLP tiene el mejor MAPE global (41.39%), pero LSTM tiene mejor RMSE.

---

**Lag = 28 días (ventana muy larga)**

| Modelo | RMSE | MAPE | Std RMSE |
|--------|------|------|----------|
| LSTM | 0.3772 | 49.53% | 0.0850 |
| MLP | 0.3786 | 51.92% | 0.0825 |
| RNN | 0.3941 | 51.65% | 0.0868 |

**Conclusión:** Todos los modelos degradan severamente. No recomendado.

---

**Estabilidad entre folds (std RMSE)**

| Modelo | Lag más estable | Std RMSE | Interpretación |
|--------|----------------|----------|----------------|
| LSTM | 7 | 0.0115 | Muy estable |
| RNN | 7 | 0.0111 | Muy estable |
| MLP | 7 | 0.0128 | Estable |
| LSTM | 14 | 0.0252 | Moderada |
| MLP | 21 | 0.0208 | Buena |
| **Todos** | **28** | **>0.08** | Inestable |

**Observación:** La estabilidad disminuye al aumentar el lag, alcanzando su punto crítico en lag=28.

---

**Test de BDS - Todos pasan**

| Métrica | Resultado |
|---------|-----------|
| BDS p mean | 0.942 - 0.977 (todos > 0.05) |
| BDS p>0.05 | 5/5 folds (100%) |

**Conclusión:** Los residuos de todos los modelos y configuraciones se comportan como ruido blanco.

---

**Resumen por objetivo de negocio**

| Si tu prioridad es... | Elegir | Configuración | Valor |
|----------------------|--------|---------------|-------|
| **Mínimo RMSE** | LSTM | lag=14 | 0.2200 |
| **Mínimo MAPE** | MLP | lag=21 | 41.39% |
| **Máxima estabilidad** | LSTM o RNN | lag=7 | std ~0.011 |
| **Balance RMSE + MAPE** | LSTM | lag=14 o 21 | 0.2200 / 42.3% |
| **Simplicidad** | MLP | lag=14 | 0.2245 / 45.34% |

## 11. Selección del mejor modelo global y exportación para despliegue en producción

Se identifica el mejor modelo global entre todas las combinaciones evaluadas de tipo de modelo, lag y fold, utilizando como criterio el menor RMSE en el conjunto de prueba. Se recorren sistemáticamente los resultados almacenados en `results_all` para localizar la configuración óptima. Una vez identificada, se recuperan los splits correspondientes a ese lag y fold, la arquitectura y máscara de dropout de la mejor configuración por tipo de modelo, y se reentrena el modelo desde cero con los datos de entrenamiento y validación completos (sin early stopping en este paso final, o con early stopping para restaurar mejores pesos). Se aplica el escalado estándar ajustado exclusivamente sobre el conjunto de entrenamiento. Para el modelo LSTM, se fuerza la ejecución en CPU debido a limitaciones de compatibilidad con DirectML. Finalmente, se exportan los artefactos necesarios para el despliegue en el directorio `app/`: el modelo en formato Keras (`model_keras.keras`), los escaladores de features y target junto con los metadatos de configuración (`scalers.joblib`), y un archivo pickle (`results.pkl`) que contiene todos los resultados del experimento, las mejores configuraciones, el modelo global óptimo y los metadatos para trazabilidad. Este conjunto de artefactos permite realizar inferencia en producción de manera reproducible y sin necesidad de reentrenar el modelo.

In [ ]:
import contextlib

best_global = {'rmse': np.inf, 'mtype': None, 'lag': None, 'fold': None}
for mtype in ['MLP', 'RNN', 'LSTM']:
    for lag in LAGS_LIST:
        for fold, r in results_all[mtype][lag].items():
            if r['rmse_test'] < best_global['rmse']:
                best_global = {'rmse': r['rmse_test'], 'mtype': mtype, 'lag': lag, 'fold': fold}

print(f"Mejor modelo: {best_global['mtype']} | lag={best_global['lag']}d | "
      f"fold={best_global['fold']} | RMSE={best_global['rmse']:.4f}")

bm = best_global
sp = splits[bm['lag']]
arch_b  = best_configs[bm['mtype']]['arquitectura']
dmask_b = best_configs[bm['mtype']]['dropout_mask']

X_tr_r  = sp['X'][bm['fold']];   y_tr_r  = sp['y'][bm['fold']]
X_val_r = sp['Xcv'][bm['fold']]; y_val_r = sp['ycv'][bm['fold']]
sx = StandardScaler().fit(X_tr_r); sy = StandardScaler().fit(y_tr_r)

if bm['mtype'] in ('RNN', 'LSTM'):
    X_in     = sx.transform(X_tr_r).reshape(-1, bm['lag'], 1)
    X_val_in = sx.transform(X_val_r).reshape(-1, bm['lag'], 1)
else:
    X_in     = sx.transform(X_tr_r)
    X_val_in = sx.transform(X_val_r)

tf.keras.backend.clear_session(); tf.random.set_seed(42)
if   bm['mtype'] == 'MLP':  best_model = build_mlp(arch_b, dmask_b, bm['lag'], N_STEPS_FORECAST)
elif bm['mtype'] == 'RNN':  best_model = build_rnn(arch_b, dmask_b, bm['lag'], N_STEPS_FORECAST)
else:                        best_model = build_lstm(arch_b, dmask_b, bm['lag'], N_STEPS_FORECAST)

ctx = tf.device('/CPU:0') if bm['mtype'] == 'LSTM' else contextlib.nullcontext()
with ctx:
    best_model.fit(X_in, sy.transform(y_tr_r),
                   validation_data=(X_val_in, sy.transform(y_val_r)),
                   epochs=EPOCHS, batch_size=BATCH_SIZE,
                   callbacks=[tf.keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True)],
                   verbose=0)

best_model.save('app/model_keras.keras')
joblib.dump({'scaler_x': sx, 'scaler_y': sy, 'lag': bm['lag'],
             'model_type': bm['mtype'], 'best_global': best_global}, 'app/scalers.joblib')

with open('results.pkl', 'wb') as f:
    pickle.dump({
        'results_all':    results_all,
        'best_configs':   best_configs,
        'best_global':    best_global,
        'lags_list':      LAGS_LIST,
        'n_steps_forecast': N_STEPS_FORECAST,
        'target_col':     TARGET_COL,
        'time_series':    time_series,
        'dates':          dates,
    }, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"\napp/model_keras.keras guardado")
print(f"app/scalers.joblib guardado")
print(f"results.pkl guardado")

Mejor modelo: LSTM | lag=21d | fold=4 | RMSE=0.1896

✅ app/model_keras.keras guardado
✅ app/scalers.joblib guardado
✅ results.pkl guardado
